# ZuCo — neutral words and sentence-context controls
**CPU → Runtime → Run all**, then allow Drive mounting. Requires the completed and audited `v4_word_erp_compact` result.

This adds two separate neutral-word controls: **neutral words in original source-rated neutral reviews** (137 SR sentences) and **neutral words in strictly neutral agent-reviewed contexts**. An overall neutral review can still include emotional words or mixed praise/criticism; these two definitions are not interchangeable. Strict context labels cover all 1,037 source sentences, across SR/NR/TSR. The strict SR subset has only two sentences and cannot support a three-word-class comparison; coverage limits are reported explicitly.

The previous pooled class means remain unchanged. This notebook reuses their accepted word IDs and onset samples, reloads verified source EEG, and reconstructs only epochs needed for the new controls. This takes longer than the saved-results review. Existing source files are reused; missing sources download in Colab one at a time and are removed from temporary Colab storage after use. No source data is downloaded to this Mac or copied into result folders.

All new results stay in `Thesis/Results/word_sentiment_eeg/v4_word_erp_compact/analysis/neutral_context_v1/`: per-reader compact means and event memberships, counts, matched-reader comparison curves, reports and a portable ZIP. Estimate roughly 100–250 MB additional Drive storage including the ZIP; the existing 1 GiB safety margin is checked. No individual-trial EEG archive is saved. Valid checkpoints resume without adding the same event twice to a given group. A word may legitimately belong to two different control definitions, explicitly recorded as two memberships, never two copies in the same average.

These are descriptive fixation-related potentials; context subsetting alone does not isolate word sentiment, and no overlap deconvolution is performed.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
from IPython.display import display, Markdown, Image

drive.mount('/content/drive')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg-context')
BASE = THESIS / 'Results/word_sentiment_eeg/v4_word_erp_compact'
ANALYSIS = BASE / 'analysis/neutral_context_v1'
if not (BASE / 'audit/integrity.json').is_file():
    raise FileNotFoundError('Run review.ipynb first to verify the saved base results. No extraction started.')
def git(*args):
    subprocess.run(['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str,args)], check=True, env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone', '--depth', '1', 'https://github.com/parmisbathayan/word-sentiment-eeg.git', REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary context checkout has edits. Preserve those before updating.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')],check=True)
print('Code version:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'run_zuco_context_controls.py'),'--thesis-root',str(THESIS),'--allow-cloud-download']
run_with_progress(command, ANALYSIS/'runtime_status.json', activity='Rebuild selected ZuCo context controls using saved word onsets')


In [ ]:
import pandas as pd
print('Saved new controls:', ANALYSIS)
display(Markdown((ANALYSIS/'REPORT.md').read_text()))
display(pd.read_csv(ANALYSIS/'comparison_coverage.csv'))
display(pd.read_csv(ANALYSIS/'cohort_counts.csv'))
for path in sorted((ANALYSIS/'plots').glob('*.png')):
    display(Image(filename=str(path)))
print('Portable results:', ANALYSIS/'context_results.zip')
